In [2]:
import os
import math
import logging
from dataclasses import dataclass
from datetime import datetime, timezone
from pathlib import Path
from typing import Optional, Sequence, Iterator
from contextlib import contextmanager

import psycopg
from psycopg.rows import dict_row
from dotenv import load_dotenv

In [3]:
# ✅ CARREGAMENTO DE BIBLIOTECAS OBRIGATÓRIAS
try:
    import pandas as pd
    import pyarrow as pa
    import pyarrow.parquet as pq
    import s3fs
    print("✅ Todas as bibliotecas obrigatórias carregadas com sucesso!")
except ImportError as e:
    raise ImportError(f"❌ Erro ao carregar bibliotecas: {e}")

# ✅ MUDANÇA DE DIRETÓRIO PARA CARREGAR .ENV CORRETO
os.chdir("/home/Projetos/Docker/Orquestradores/airflow/dags/Sost/scripts/Datalake/Bronze")
print(f"✅ Diretório atual: {os.getcwd()}")

# ✅ SETUP DE LOGGING
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s - %(levelname)s - %(message)s"
)
logger = logging.getLogger(__name__)

# ===========================
# DATACLASSES PARA CONFIGURAÇÃO
# ===========================

@dataclass(frozen=True)
class PostgresConfig:
    """Configuração PostgreSQL"""
    host: str
    port: int
    database: str
    user: str
    password: str

@dataclass(frozen=True)
class MinioConfig:
    """Configuração MinIO"""
    endpoint: str
    bucket: str
    access_key: str
    secret_key: str

@dataclass(frozen=True)
class TableSpec:
    """Especificação de tabela"""
    schema: str
    name: str
    chunk_rows: int = 200_000

# ===========================
# CARREGAMENTO DE CONFIGURAÇÃO
# ===========================

def load_postgres_config() -> PostgresConfig:
    """Carrega configuração PostgreSQL do .env"""
    load_dotenv(".env")
    
    host = os.getenv("POSTGRES_DW_HOST", "").strip()
    port = int(os.getenv("POSTGRES_DW_PORT", "5432").strip()) if os.getenv("POSTGRES_DW_PORT") else 5432
    database = os.getenv("POSTGRES_DW_DB", "").strip()
    user = os.getenv("POSTGRES_DW_USER", "").strip()
    password = os.getenv("POSTGRES_DW_PASSWORD", "").strip()
    
    if not (host and database and user and password):
        raise ValueError(
            f"❌ Configuração PostgreSQL incompleta:\n"
            f"  HOST: {host if host else 'VAZIO'} | PORT: {port}\n"
            f"  DATABASE: {database if database else 'VAZIO'} | USER: {user if user else 'VAZIO'}"
        )
    
    return PostgresConfig(
        host=host,
        port=port,
        database=database,
        user=user,
        password=password
    )

def load_minio_config() -> MinioConfig:
    """Carrega configuração MinIO do .env"""
    load_dotenv(".env")
    
    endpoint = os.getenv("MINIO_ENDPOINT", "").strip()
    bucket = os.getenv("MINIO_BUCKET_SOST", "").strip()
    access_key = os.getenv("MINIO_ROOT_USER", "").strip()
    secret_key = os.getenv("MINIO_ROOT_PASSWORD", "").strip()
    
    if not (endpoint and bucket and access_key and secret_key):
        raise ValueError(
            f"❌ Configuração MinIO incompleta:\n"
            f"  ENDPOINT: {endpoint[:20]}... | BUCKET: {bucket}\n"
            f"  ACCESS_KEY: {access_key[:5]}... | SECRET_KEY: {secret_key[:5] if secret_key else 'VAZIO'}..."
        )
    
    return MinioConfig(
        endpoint=endpoint,
        bucket=bucket,
        access_key=access_key,
        secret_key=secret_key
    )

# ===========================
# CONEXÕES
# ===========================

def minio_fs(cfg: MinioConfig) -> s3fs.S3FileSystem:
    """Cria conexão S3FileSystem com MinIO"""
    return s3fs.S3FileSystem(
        key=cfg.access_key,
        secret=cfg.secret_key,
        client_kwargs={"endpoint_url": cfg.endpoint},
        config_kwargs={"signature_version": "s3v4"},
        use_ssl=False,
    )

def pg_conn_str(cfg: PostgresConfig) -> str:
    """Constrói string de conexão PostgreSQL"""
    from urllib.parse import quote
    safe_password = quote(cfg.password, safe='')
    return (
        f"postgresql://{cfg.user}:{safe_password}@{cfg.host}:{cfg.port}/{cfg.database}"
    )

@contextmanager
def pg_connection(cfg: PostgresConfig):
    """Context manager para conexão PostgreSQL"""
    conn = psycopg.connect(pg_conn_str(cfg), autocommit=False)
    try:
        yield conn
    finally:
        conn.close()

# ===========================
# FUNÇÕES DE UTILIDADE
# ===========================

def build_select_sql(table_spec: TableSpec) -> str:
    """Constrói SQL de seleção"""
    return f"SELECT * FROM {table_spec.schema}.{table_spec.name};"

def count_rows(conn, table_spec: TableSpec) -> int:
    """Conta linhas na tabela"""
    sql = f"SELECT COUNT(*) as cnt FROM {table_spec.schema}.{table_spec.name};"
    with conn.cursor() as cur:
        cur.execute(sql)
        return cur.fetchone()[0]

def write_parquet_to_minio(
    df: pd.DataFrame,
    fs: s3fs.S3FileSystem,
    bucket: str,
    schema: str,
    table: str,
    chunk_idx: int,
    ingestion_dt: datetime,
    filename: str = None
) -> str:
    """Escreve parquet direto no MinIO sem passar pelo disco local"""
    df["_ingestion_ts_utc"] = ingestion_dt.isoformat()
    df["_ingestion_date"] = ingestion_dt.date().isoformat()

    # ✅ Usa nome customizado ou gera com parte numerada
    if filename:
        file_part = filename
    else:
        file_part = f"part-{chunk_idx:05d}.parquet"
    
    s3_path = f"s3://{bucket}/bronze/003_bronze_pcsupervisor/{file_part}"

    table_pa = pa.Table.from_pandas(df, preserve_index=False)
    
    # ✅ Escreve direto no S3/MinIO (sem cache local)
    with fs.open(s3_path, "wb") as f:
        pq.write_table(table_pa, f, compression="snappy")
    
    logger.info(f"✅ Arquivo escrito: {s3_path}")
    return s3_path

# ===========================
# FUNÇÃO PRINCIPAL DE EXTRAÇÃO
# ===========================

def extract_to_minio(
    conn: psycopg.Connection,
    fs: s3fs.S3FileSystem,
    table_spec: TableSpec,
    minio_cfg: MinioConfig,
    ingestion_dt: datetime
) -> dict:
    """Extrai tabela PostgreSQL → MinIO usando cursor servidor-side"""
    
    logger.info(f"🚀 Iniciando extração: {table_spec.schema}.{table_spec.name}")
    
    # Conta linhas totais
    total_rows = count_rows(conn, table_spec)
    logger.info(f"📊 Total de linhas: {total_rows}")
    
    sql = build_select_sql(table_spec)
    stats = {
        "total_rows": total_rows,
        "chunks_created": 0,
        "files_written": 0,
        "bytes_written": 0
    }
    
    # ✅ CURSOR SERVIDOR-SIDE: Não carrega tudo em memória
    with conn.transaction():
        with conn.cursor(name="cur_extraction", row_factory=dict_row) as cur:
            cur.itersize = table_spec.chunk_rows  # 200k por fetch
            cur.execute(sql)
            
            chunk_idx = 0
            while True:
                # Fetch de 200k linhas por vez
                rows = cur.fetchmany(table_spec.chunk_rows)
                if not rows:
                    break
                
                chunk_idx += 1
                logger.info(f"📦 Processando chunk {chunk_idx} ({len(rows)} linhas)...")
                
                # Converte para DataFrame
                df = pd.DataFrame(rows)
                
                # Escreve direto para MinIO
                s3_path = write_parquet_to_minio(
                    df=df,
                    fs=fs,
                    bucket=minio_cfg.bucket,
                    schema=table_spec.schema,
                    table=table_spec.name,
                    chunk_idx=chunk_idx,
                    ingestion_dt=ingestion_dt
                )
                
                stats["chunks_created"] += 1
                stats["files_written"] += 1
    
    logger.info(f"✅ Extração completa!")
    logger.info(f"📈 Estatísticas: {stats}")
    return stats

# ===========================
# EXECUÇÃO PRINCIPAL
# ===========================

def main():
    """Orquestra a extração completa"""
    try:
        logger.info("="*60)
        logger.info("🎯 INICIANDO EXTRAÇÃO BRONZE: pcsuperv")
        logger.info("="*60)
        
        # Carrega configurações
        pg_cfg = load_postgres_config()
        minio_cfg = load_minio_config()
        logger.info(f"✅ Configurações carregadas")
        logger.info(f"   PostgreSQL: {pg_cfg.host}:{pg_cfg.port}/{pg_cfg.database}")
        logger.info(f"   MinIO: {minio_cfg.endpoint} / {minio_cfg.bucket}")
        
        # Cria conexão MinIO
        fs = minio_fs(minio_cfg)
        logger.info(f"✅ Conexão MinIO estabelecida")
        
        # Define tabela a extrair
        table_spec = TableSpec(schema="public", name="pcsuperv", chunk_rows=200_000)
        
        # Conecta ao PostgreSQL e extrai
        with pg_connection(pg_cfg) as conn:
            stats = extract_to_minio(
                conn=conn,
                fs=fs,
                table_spec=table_spec,
                minio_cfg=minio_cfg,
                ingestion_dt=datetime.now(timezone.utc)
            )
        
        logger.info("="*60)
        logger.info("✅ SUCESSO!")
        logger.info("="*60)
        return stats
        
    except Exception as e:
        logger.error(f"❌ ERRO: {e}", exc_info=True)
        raise

# ✅ EXECUTA EXTRAÇÃO
if __name__ == "__main__":
    result = main()
    print("\n" + "="*60)
    print("RESULTADO FINAL:")
    print("="*60)
    for key, value in result.items():
        print(f"{key}: {value}")
    print("="*60)

2026-02-10 00:05:09,882 - INFO - ============================================================
2026-02-10 00:05:09,883 - INFO - 🎯 INICIANDO EXTRAÇÃO BRONZE: pcsuperv
2026-02-10 00:05:09,883 - INFO - ============================================================
2026-02-10 00:05:09,886 - INFO - ✅ Configurações carregadas
2026-02-10 00:05:09,886 - INFO -    PostgreSQL: 209.50.228.232:5433/ERP
2026-02-10 00:05:09,886 - INFO -    MinIO: http://209.50.228.232:9000 / datalake-sost
2026-02-10 00:05:09,888 - INFO - ✅ Conexão MinIO estabelecida


✅ Todas as bibliotecas obrigatórias carregadas com sucesso!
✅ Diretório atual: /home/Projetos/Docker/Orquestradores/airflow/dags/Sost/scripts/Datalake/Bronze


2026-02-10 00:05:10,540 - INFO - 🚀 Iniciando extração: public.pcsuperv
2026-02-10 00:05:10,814 - INFO - 📊 Total de linhas: 47
2026-02-10 00:05:11,331 - INFO - 📦 Processando chunk 1 (47 linhas)...
2026-02-10 00:05:11,950 - INFO - ✅ Arquivo escrito: s3://datalake-sost/bronze/003_bronze_pcsupervisor/part-00001.parquet
2026-02-10 00:05:12,338 - INFO - ✅ Extração completa!
2026-02-10 00:05:12,339 - INFO - 📈 Estatísticas: {'total_rows': 47, 'chunks_created': 1, 'files_written': 1, 'bytes_written': 0}
2026-02-10 00:05:12,340 - INFO - ============================================================
2026-02-10 00:05:12,340 - INFO - ✅ SUCESSO!
2026-02-10 00:05:12,340 - INFO - ============================================================



RESULTADO FINAL:
total_rows: 47
chunks_created: 1
files_written: 1
bytes_written: 0


In [ ]:
# ===========================
# VALIDAÇÃO: LER ARQUIVO DO MINIO
# ===========================

def read_parquet_from_minio(
    fs: s3fs.S3FileSystem,
    s3_path: str,
    sample_rows: int = 5
) -> pd.DataFrame:
    """Lê arquivo parquet do MinIO e retorna amostra"""
    logger.info(f"📖 Lendo arquivo: {s3_path}")
    
    with fs.open(s3_path, "rb") as f:
        table = pq.read_table(f)
        df = table.to_pandas()
    
    logger.info(f"✅ Arquivo lido com sucesso!")
    logger.info(f"📊 Total de linhas: {len(df)}")
    logger.info(f"📋 Colunas: {', '.join(df.columns.tolist())}")
    
    return df

# ===========================
# TESTE: VALIDAR ARQUIVO NO MINIO
# ===========================

def validate_extraction():
    """Valida a extração lendo o arquivo do MinIO"""
    try:
        logger.info("="*60)
        logger.info("🔍 VALIDAÇÃO: Lendo parquet do MinIO")
        logger.info("="*60)
        
        # Carrega configurações
        minio_cfg = load_minio_config()
        fs = minio_fs(minio_cfg)
        
        # Caminho do arquivo em /bronze/003_bronze_pcsupervisor/
        s3_path = f"s3://{minio_cfg.bucket}/bronze/003_bronze_pcsupervisor/part-00001.parquet"
        
        # Lê arquivo
        df = read_parquet_from_minio(fs, s3_path)
        
        logger.info("\n📋 PREVIEW DOS DADOS:")
        logger.info(f"\n{df.head(10).to_string()}")
        
        logger.info(f"\n✅ VALIDAÇÃO COMPLETA!")
        
        return df
        
    except Exception as e:
        logger.error(f"❌ ERRO NA VALIDAÇÃO: {e}", exc_info=True)
        raise

# ✅ EXECUTA VALIDAÇÃO
if __name__ == "__main__":
    df_validation = validate_extraction()
    
    print("\n" + "="*60)
    print("RESUMO DA VALIDAÇÃO:")
    print("="*60)
    print(f"Linhas totais: {len(df_validation)}")
    print(f"Colunas: {len(df_validation.columns)}")
    print(f"Primeiras linhas:")
    print(df_validation.head())
    print("="*60)

2026-02-08 01:35:08,107 - INFO - ============================================================
2026-02-08 01:35:08,108 - INFO - 🔍 VALIDAÇÃO: Lendo parquet do MinIO
2026-02-08 01:35:08,108 - INFO - ============================================================
2026-02-08 01:35:08,110 - INFO - 📖 Lendo arquivo: s3://datalake-sost/bronze/003_bronze_pcsupervisor.parquet
2026-02-08 01:35:08,506 - INFO - ✅ Arquivo lido com sucesso!
2026-02-08 01:35:08,507 - INFO - 📊 Total de linhas: 47
2026-02-08 01:35:08,508 - INFO - 📋 Colunas: codsupervisor, nome, regional, cod_cadrca, posicao, percpartvendaprev, percmargemprev, codgerente, tiposupervisor, percomissao, dtadmissao, dtdemissao, cpf, codcoordenador, email, vlcorrente, vllimcred, usadebcred, dt_vmais, _ingestion_ts_utc, _ingestion_date
2026-02-08 01:35:08,508 - INFO - 
📋 PREVIEW DOS DADOS:
2026-02-08 01:35:08,513 - INFO - 
   codsupervisor                             nome  regional  cod_cadrca posicao percpartvendaprev percmargemprev  codgerente t


RESUMO DA VALIDAÇÃO:
Linhas totais: 47
Colunas: 21
Primeiras linhas:
   codsupervisor                           nome  regional  cod_cadrca posicao  \
0             11              SUPERVISOR VAREJO         1        56.0       I   
1              6                  MELISSA DIOGO         1        12.0       I   
2              7                     JANE REGIS         1        11.0       I   
3              1      SUPERVISOR PARAMETRIZACAO         1         1.0       I   
4              2  ALINE FERREIRA DE JESUS GOMES         1         2.0       I   

  percpartvendaprev percmargemprev  codgerente tiposupervisor percomissao  \
0              None           None           3              I        0.25   
1              None           None           3              I        0.30   
2              None           None           2              I        None   
3              None           None           1              I        5.00   
4              None           None           4            

In [7]:
import os
import math
import logging
from dataclasses import dataclass
from datetime import datetime, timezone
from pathlib import Path
from typing import Optional, Sequence, Iterator
from contextlib import contextmanager

import psycopg
from psycopg.rows import dict_row
from dotenv import load_dotenv


In [8]:
# ✅ CARREGAMENTO DE BIBLIOTECAS OBRIGATÓRIAS
try:
    import pandas as pd
    import pyarrow as pa
    import pyarrow.parquet as pq
    import s3fs
    print("✅ Todas as bibliotecas obrigatórias carregadas com sucesso!")
except ImportError as e:
    raise ImportError(f"❌ Erro ao carregar bibliotecas: {e}")

# ✅ MUDANÇA DE DIRETÓRIO PARA CARREGAR .ENV CORRETO
os.chdir("/home/Projetos/Docker/Orquestradores/airflow/dags/Sost/scripts/Datalake/Bronze")
print(f"✅ Diretório atual: {os.getcwd()}")

# ✅ SETUP DE LOGGING
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s - %(levelname)s - %(message)s"
)
logger = logging.getLogger(__name__)

# ===========================
# DATACLASSES PARA CONFIGURAÇÃO
# ===========================

@dataclass(frozen=True)
class PostgresConfig:
    """Configuração PostgreSQL"""
    host: str
    port: int
    database: str
    user: str
    password: str

@dataclass(frozen=True)
class MinioConfig:
    """Configuração MinIO"""
    endpoint: str
    bucket: str
    access_key: str
    secret_key: str

@dataclass(frozen=True)
class TableSpec:
    """Especificação de tabela"""
    schema: str
    name: str
    chunk_rows: int = 200_000

# ===========================
# CARREGAMENTO DE CONFIGURAÇÃO
# ===========================

def load_postgres_config() -> PostgresConfig:
    """Carrega configuração PostgreSQL do .env"""
    load_dotenv(".env")
    
    host = os.getenv("POSTGRES_DW_HOST", "").strip()
    port = int(os.getenv("POSTGRES_DW_PORT", "5432").strip()) if os.getenv("POSTGRES_DW_PORT") else 5432
    database = os.getenv("POSTGRES_DW_DB", "").strip()
    user = os.getenv("POSTGRES_DW_USER", "").strip()
    password = os.getenv("POSTGRES_DW_PASSWORD", "").strip()
    
    if not (host and database and user and password):
        raise ValueError(
            f"❌ Configuração PostgreSQL incompleta:\n"
            f"  HOST: {host if host else 'VAZIO'} | PORT: {port}\n"
            f"  DATABASE: {database if database else 'VAZIO'} | USER: {user if user else 'VAZIO'}"
        )
    
    return PostgresConfig(
        host=host,
        port=port,
        database=database,
        user=user,
        password=password
    )

def load_minio_config() -> MinioConfig:
    """Carrega configuração MinIO do .env"""
    load_dotenv(".env")
    
    endpoint = os.getenv("MINIO_ENDPOINT", "").strip()
    bucket = os.getenv("MINIO_BUCKET_SOST", "").strip()
    access_key = os.getenv("MINIO_ROOT_USER", "").strip()
    secret_key = os.getenv("MINIO_ROOT_PASSWORD", "").strip()
    
    if not (endpoint and bucket and access_key and secret_key):
        raise ValueError(
            f"❌ Configuração MinIO incompleta:\n"
            f"  ENDPOINT: {endpoint[:20]}... | BUCKET: {bucket}\n"
            f"  ACCESS_KEY: {access_key[:5]}... | SECRET_KEY: {secret_key[:5] if secret_key else 'VAZIO'}..."
        )
    
    return MinioConfig(
        endpoint=endpoint,
        bucket=bucket,
        access_key=access_key,
        secret_key=secret_key
    )

# ===========================
# CONEXÕES
# ===========================

def minio_fs(cfg: MinioConfig) -> s3fs.S3FileSystem:
    """Cria conexão S3FileSystem com MinIO"""
    return s3fs.S3FileSystem(
        key=cfg.access_key,
        secret=cfg.secret_key,
        client_kwargs={"endpoint_url": cfg.endpoint},
        config_kwargs={"signature_version": "s3v4"},
        use_ssl=False,
    )

def pg_conn_str(cfg: PostgresConfig) -> str:
    """Constrói string de conexão PostgreSQL"""
    from urllib.parse import quote
    safe_password = quote(cfg.password, safe='')
    return (
        f"postgresql://{cfg.user}:{safe_password}@{cfg.host}:{cfg.port}/{cfg.database}"
    )

@contextmanager
def pg_connection(cfg: PostgresConfig):
    """Context manager para conexão PostgreSQL"""
    conn = psycopg.connect(pg_conn_str(cfg), autocommit=False)
    try:
        yield conn
    finally:
        conn.close()

# ===========================
# FUNÇÕES DE UTILIDADE
# ===========================

def build_select_sql(table_spec: TableSpec) -> str:
    """Constrói SQL de seleção"""
    return f"SELECT * FROM {table_spec.schema}.{table_spec.name};"

def count_rows(conn, table_spec: TableSpec) -> int:
    """Conta linhas na tabela"""
    sql = f"SELECT COUNT(*) as cnt FROM {table_spec.schema}.{table_spec.name};"
    with conn.cursor() as cur:
        cur.execute(sql)
        return cur.fetchone()[0]

def write_parquet_to_minio(
    df: pd.DataFrame,
    fs: s3fs.S3FileSystem,
    bucket: str,
    schema: str,
    table: str,
    chunk_idx: int,
    ingestion_dt: datetime,
    filename: str = None
) -> str:
    """Escreve parquet direto no MinIO sem passar pelo disco local"""
    df["_ingestion_ts_utc"] = ingestion_dt.isoformat()
    df["_ingestion_date"] = ingestion_dt.date().isoformat()

    # ✅ Usa nome customizado ou gera com parte numerada
    if filename:
        file_part = filename
    else:
        file_part = f"part-{chunk_idx:05d}.parquet"
    
    s3_path = f"s3://{bucket}/bronze/{file_part}"

    table_pa = pa.Table.from_pandas(df, preserve_index=False)
    
    # ✅ Escreve direto no S3/MinIO (sem cache local)
    with fs.open(s3_path, "wb") as f:
        pq.write_table(table_pa, f, compression="snappy")
    
    logger.info(f"✅ Arquivo escrito: {s3_path}")
    return s3_path

# ===========================
# FUNÇÃO PRINCIPAL DE EXTRAÇÃO
# ===========================

def extract_to_minio(
    conn: psycopg.Connection,
    fs: s3fs.S3FileSystem,
    table_spec: TableSpec,
    minio_cfg: MinioConfig,
    ingestion_dt: datetime
) -> dict:
    """Extrai tabela PostgreSQL → MinIO usando cursor servidor-side"""
    
    logger.info(f"🚀 Iniciando extração: {table_spec.schema}.{table_spec.name}")
    
    # Conta linhas totais
    total_rows = count_rows(conn, table_spec)
    logger.info(f"📊 Total de linhas: {total_rows}")
    
    sql = build_select_sql(table_spec)
    stats = {
        "total_rows": total_rows,
        "chunks_created": 0,
        "files_written": 0,
        "bytes_written": 0
    }
    
    # ✅ CURSOR SERVIDOR-SIDE: Não carrega tudo em memória
    with conn.transaction():
        with conn.cursor(name="cur_extraction", row_factory=dict_row) as cur:
            cur.itersize = table_spec.chunk_rows  # 200k por fetch
            cur.execute(sql)
            
            chunk_idx = 0
            while True:
                # Fetch de 200k linhas por vez
                rows = cur.fetchmany(table_spec.chunk_rows)
                if not rows:
                    break
                
                chunk_idx += 1
                logger.info(f"📦 Processando chunk {chunk_idx} ({len(rows)} linhas)...")
                
                # Converte para DataFrame
                df = pd.DataFrame(rows)
                
                # Escreve direto para MinIO
                s3_path = write_parquet_to_minio(
                    df=df,
                    fs=fs,
                    bucket=minio_cfg.bucket,
                    schema=table_spec.schema,
                    table=table_spec.name,
                    chunk_idx=chunk_idx,
                    ingestion_dt=ingestion_dt,
                    filename="003_bronze_pcsupervisor.parquet"
                )
                
                stats["chunks_created"] += 1
                stats["files_written"] += 1
                # stats["bytes_written"] += df.memory_usage(deep=True).sum()
    
    logger.info(f"✅ Extração completa!")
    logger.info(f"📈 Estatísticas: {stats}")
    return stats

# ===========================
# EXECUÇÃO PRINCIPAL
# ===========================

def main():
    """Orquestra a extração completa"""
    try:
        logger.info("="*60)
        logger.info("🎯 INICIANDO EXTRAÇÃO BRONZE: pcsuperv")
        logger.info("="*60)
        
        # Carrega configurações
        pg_cfg = load_postgres_config()
        minio_cfg = load_minio_config()
        logger.info(f"✅ Configurações carregadas")
        logger.info(f"   PostgreSQL: {pg_cfg.host}:{pg_cfg.port}/{pg_cfg.database}")
        logger.info(f"   MinIO: {minio_cfg.endpoint} / {minio_cfg.bucket}")
        
        # Cria conexão MinIO
        fs = minio_fs(minio_cfg)
        logger.info(f"✅ Conexão MinIO estabelecida")
        
        # Define tabela a extrair
        table_spec = TableSpec(schema="public", name="pcsuperv", chunk_rows=200_000)
        
        # Conecta ao PostgreSQL e extrai
        with pg_connection(pg_cfg) as conn:
            stats = extract_to_minio(
                conn=conn,
                fs=fs,
                table_spec=table_spec,
                minio_cfg=minio_cfg,
                ingestion_dt=datetime.now(timezone.utc)
            )
        
        logger.info("="*60)
        logger.info("✅ SUCESSO!")
        logger.info("="*60)
        return stats
        
    except Exception as e:
        logger.error(f"❌ ERRO: {e}", exc_info=True)
        raise

# ✅ EXECUTA EXTRAÇÃO
if __name__ == "__main__":
    result = main()
    print("\n" + "="*60)
    print("RESULTADO FINAL:")
    print("="*60)
    for key, value in result.items():
        print(f"{key}: {value}")
    print("="*60)

2026-02-08 01:28:25,311 - INFO - ============================================================
2026-02-08 01:28:25,312 - INFO - 🎯 INICIANDO EXTRAÇÃO BRONZE: pcsuperv
2026-02-08 01:28:25,313 - INFO - ============================================================
2026-02-08 01:28:25,316 - INFO - ✅ Configurações carregadas
2026-02-08 01:28:25,316 - INFO -    PostgreSQL: 209.50.228.232:5433/ERP
2026-02-08 01:28:25,316 - INFO -    MinIO: http://209.50.228.232:9000 / datalake-sost
2026-02-08 01:28:25,317 - INFO - ✅ Conexão MinIO estabelecida


✅ Todas as bibliotecas obrigatórias carregadas com sucesso!
✅ Diretório atual: /home/Projetos/Docker/Orquestradores/airflow/dags/Sost/scripts/Datalake/Bronze


2026-02-08 01:28:25,964 - INFO - 🚀 Iniciando extração: public.pcsuperv
2026-02-08 01:28:26,223 - INFO - 📊 Total de linhas: 47
2026-02-08 01:28:26,739 - INFO - 📦 Processando chunk 1 (47 linhas)...
2026-02-08 01:28:27,272 - INFO - ✅ Arquivo escrito: s3://datalake-sost/bronze/003_bronze_pcsupervisor.parquet
2026-02-08 01:28:27,659 - INFO - ✅ Extração completa!
2026-02-08 01:28:27,660 - INFO - 📈 Estatísticas: {'total_rows': 47, 'chunks_created': 1, 'files_written': 1, 'bytes_written': 0}
2026-02-08 01:28:27,661 - INFO - ============================================================
2026-02-08 01:28:27,661 - INFO - ✅ SUCESSO!
2026-02-08 01:28:27,661 - INFO - ============================================================



RESULTADO FINAL:
total_rows: 47
chunks_created: 1
files_written: 1
bytes_written: 0


In [6]:
# ===========================
# VALIDAÇÃO: LER ARQUIVO DO MINIO
# ===========================

def read_parquet_from_minio(
    fs: s3fs.S3FileSystem,
    s3_path: str,
    sample_rows: int = 5
) -> pd.DataFrame:
    """Lê arquivo parquet do MinIO e retorna amostra"""
    logger.info(f"📖 Lendo arquivo: {s3_path}")
    
    with fs.open(s3_path, "rb") as f:
        table = pq.read_table(f)
        df = table.to_pandas()
    
    logger.info(f"✅ Arquivo lido com sucesso!")
    logger.info(f"📊 Total de linhas: {len(df)}")
    logger.info(f"📋 Colunas: {', '.join(df.columns.tolist())}")
    
    return df

# ===========================
# TESTE: VALIDAR ARQUIVO NO MINIO
# ===========================

def validate_extraction():
    """Valida a extração lendo o arquivo do MinIO"""
    try:
        logger.info("="*60)
        logger.info("🔍 VALIDAÇÃO: Lendo parquet do MinIO")
        logger.info("="*60)
        
        # Carrega configurações
        minio_cfg = load_minio_config()
        fs = minio_fs(minio_cfg)
        
        # Caminho do arquivo
        s3_path = f"s3://{minio_cfg.bucket}/bronze/003_bronze_pcsupervisor.parquet"
        
        # Lê arquivo
        df = read_parquet_from_minio(fs, s3_path)
        
        logger.info("\n📋 PREVIEW DOS DADOS:")
        logger.info(f"\n{df.head(10).to_string()}")
        
        logger.info(f"\n✅ VALIDAÇÃO COMPLETA!")
        
        return df
        
    except Exception as e:
        logger.error(f"❌ ERRO NA VALIDAÇÃO: {e}", exc_info=True)
        raise

# ✅ EXECUTA VALIDAÇÃO
if __name__ == "__main__":
    df_validation = validate_extraction()
    
    print("\n" + "="*60)
    print("RESUMO DA VALIDAÇÃO:")
    print("="*60)
    print(f"Linhas totais: {len(df_validation)}")
    print(f"Colunas: {len(df_validation.columns)}")
    print(f"Primeiras linhas:")
    print(df_validation.head())
    print("="*60)

2026-02-08 01:27:58,183 - INFO - ============================================================
2026-02-08 01:27:58,184 - INFO - 🔍 VALIDAÇÃO: Lendo parquet do MinIO
2026-02-08 01:27:58,184 - INFO - ============================================================
2026-02-08 01:27:58,187 - INFO - 📖 Lendo arquivo: s3://datalake-sost/bronze/003_bronze_pcsupervisor.parquet
2026-02-08 01:27:58,594 - INFO - ✅ Arquivo lido com sucesso!
2026-02-08 01:27:58,594 - INFO - 📊 Total de linhas: 47
2026-02-08 01:27:58,595 - INFO - 📋 Colunas: codsupervisor, nome, regional, cod_cadrca, posicao, percpartvendaprev, percmargemprev, codgerente, tiposupervisor, percomissao, dtadmissao, dtdemissao, cpf, codcoordenador, email, vlcorrente, vllimcred, usadebcred, dt_vmais, _ingestion_ts_utc, _ingestion_date
2026-02-08 01:27:58,595 - INFO - 
📋 PREVIEW DOS DADOS:
2026-02-08 01:27:58,601 - INFO - 
   codsupervisor                             nome  regional  cod_cadrca posicao percpartvendaprev percmargemprev  codgerente t


RESUMO DA VALIDAÇÃO:
Linhas totais: 47
Colunas: 21
Primeiras linhas:
   codsupervisor                           nome  regional  cod_cadrca posicao  \
0             11              SUPERVISOR VAREJO         1        56.0       I   
1              6                  MELISSA DIOGO         1        12.0       I   
2              7                     JANE REGIS         1        11.0       I   
3              1      SUPERVISOR PARAMETRIZACAO         1         1.0       I   
4              2  ALINE FERREIRA DE JESUS GOMES         1         2.0       I   

  percpartvendaprev percmargemprev  codgerente tiposupervisor percomissao  \
0              None           None           3              I        0.25   
1              None           None           3              I        0.30   
2              None           None           2              I        None   
3              None           None           1              I        5.00   
4              None           None           4            